In [22]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import os 
import warnings 
warnings.filterwarnings("ignore")

# Feature Engineering 

In [23]:
#Load the cleaned dataset 
df= pd.read_csv(os.path.join("..","Processed","cleaned_data.csv"))
print(df.shape)

name_mapping= {
    'Alavés': 'Alaves',
    'Atlético Madrid': 'Atletico Madrid',
    'Betis': 'Real Betis',
    'Cádiz': 'Cadiz',
    'Málaga': 'Malaga',
    'Leganés': 'Leganes',
    'Almería': 'Almeria',
}

df['team']=df['team'].replace(name_mapping)
df['opponent']=df['opponent'].replace(name_mapping)
print(f"Unique teams: {df['team'].nunique()}")
print(f"Unique opponents: {df['opponent'].nunique()}")

team_set=set(df['team'].unique())
opponent_set=set(df['opponent'].unique())
if team_set==opponent_set:
    print("Team and Opponent names match.")
else:
    print("Mismatch in Team and Opponent names.")

#Convert date back to date time 
df['date'] = pd.to_datetime(df['date'],errors='coerce')

#sort by team and date ensure proper rolling calculations
df=df.sort_values(["team","date"]).reset_index(drop=True)
is_sorted=df.groupby("team")["date"].apply(lambda x: x.is_monotonic_increasing).all()
print(f"Data sorted by team and date: {is_sorted}")

#Check for duplicates
dupes=df.duplicated(subset=['team', 'date', 'opponent'], keep=False)
if dupes.any():
    print(f"Found Duplicates:{dupes.sum()}")
else:
    print("No Duplicates Found")

#Check for missing time or dates
print("Teams:", df['team'].nunique())
print("Date range:", df['date'].min(), "to", df['date'].max())
print("Missing dates:", df['date'].isna().sum())
print(df.head())


(4700, 29)
Unique teams: 28
Unique opponents: 28
Team and Opponent names match.
Data sorted by team and date: True
No Duplicates Found
Teams: 28
Date range: 2019-08-16 00:00:00 to 2025-09-30 00:00:00
Missing dates: 0
        date   time     comp        round  day venue result  gf  ga       opponent   xg  xga  poss  attendance           captain formation opp formation              referee  sh  sot  dist  fk  pk  pkatt  season    team  year  month day_of_week
0 2019-08-18  17:00  La Liga  Matchweek 1  Sun  Home      W   1   0        Levante  0.5  1.1  38.0     12029.0       Manu García   4-1-4-1     4-1-2-1-2           César Soto   9    2  18.6   1   0      0    2019  Alaves  2019      8      Sunday
1 2019-08-25  17:00  La Liga  Matchweek 2  Sun  Home      D   0   0       Espanyol  0.8  0.2  37.0     14567.0       Manu García   4-1-4-1         4-3-3       Eduardo Prieto  10    2  18.3   0   0      0    2019  Alaves  2019      8      Sunday
2 2019-08-31  19:00  La Liga  Matchweek 3  Sat  

# Create target variable (Outcome)

In [24]:
# Map result to numeric outcomes: W=1, D=0, L=-1
outcome_mapping={'W':1, 'D':0, 'L':-1}
df["outcome"]=df["result"].map(outcome_mapping)

print("Outcome distribution:\n")
print(df["outcome"].value_counts().sort_index())

print(f"\nWin rate: {(df["outcome"]==1).sum()/len(df)*100:.2f}%")
print(f"\nDraw rate: {(df["outcome"]==0).sum()/len(df)*100:.2f}%")
print(f"\nLoss rate: {(df["outcome"]==-1).sum()/len(df)*100:.2f}%")

Outcome distribution:

outcome
-1    1712
 0    1276
 1    1712
Name: count, dtype: int64

Win rate: 36.43%

Draw rate: 27.15%

Loss rate: 36.43%


# Creating rolling averages for the last 5 matches per team 

In [25]:
# Calculating rolling averages for key performance metrics
rolling_features = ['gf', 'ga', 'xg', 'xga', 'poss', 'sh', 'sot', 'dist', 'fk', 'pk', 'pkatt']
window_size = 5

print(f"\nCalculating {window_size}-match rolling averages for performance metrics")
print(f"Features: {rolling_features}\n")

# Initialize rolling feature columns
for feature in rolling_features:
    df[f"avg_{feature}_{window_size}"] = np.nan

# Calculating rolling avg grouped by team
# Using shift(1) to avoid data leakage from current match
for team in df['team'].unique():
    team_mask = df['team'] == team
    team_indices = df[team_mask].index
    
    for feature in rolling_features:                     
        if feature in df.columns:
            # shift(1): exclude current match (avoid leakage)
            # rolling(window=5, min_periods=1): use up to 5 prior matches, but allow fewer early on
            rolling_values = (
                df.loc[team_mask, feature]
                  .shift(1)
                  .rolling(window=window_size, min_periods=1)
                  .mean()
            )
            df.loc[team_indices, f"avg_{feature}_{window_size}"] = rolling_values.values

print("Rolling avg features created")

for feature in rolling_features:
    if feature in df.columns:
        col_name = f"avg_{feature}_{window_size}"
        missing = df[col_name].isna().sum()
        print(f" - {col_name}, Missing values={missing}")

df=df.dropna(subset=[f"avg_{feature}_{window_size}" for feature in rolling_features])
print(f"\nData shape after dropping rows with missing rolling averages: {df.shape}")
# Show first 2 matches for one team before dropping
team_sample = df[df['team'] == 'Barcelona'][['date', 'gf', 'avg_gf_5']].head(3)
print(team_sample)


Calculating 5-match rolling averages for performance metrics
Features: ['gf', 'ga', 'xg', 'xga', 'poss', 'sh', 'sot', 'dist', 'fk', 'pk', 'pkatt']

Rolling avg features created
 - avg_gf_5, Missing values=28
 - avg_ga_5, Missing values=28
 - avg_xg_5, Missing values=28
 - avg_xga_5, Missing values=28
 - avg_poss_5, Missing values=28
 - avg_sh_5, Missing values=28
 - avg_sot_5, Missing values=28
 - avg_dist_5, Missing values=28
 - avg_fk_5, Missing values=28
 - avg_pk_5, Missing values=28
 - avg_pkatt_5, Missing values=28

Data shape after dropping rows with missing rolling averages: (4672, 41)
          date  gf  avg_gf_5
744 2019-08-25   5  0.000000
745 2019-08-31   2  2.500000
746 2019-09-14   5  2.333333


# Create additional derived features 

In [26]:
#Goal difference rolling avg
df['avg_goal_diff']=df['avg_gf_5']-df['avg_ga_5']

# Expexted goal difference rolling avg
df['avg_xgoal_diff']=df['avg_xg_5']-df['avg_xga_5']

#Shot accuracy
df['avg_shot_acc']=np.where(
    df['avg_sh_5']>0,
    df['avg_sot_5']/df['avg_sh_5'],0
)
#Penalty conversion rate
df['avg_pk_acc']=np.where(
    df['avg_pkatt_5']>0,
    df['avg_pk_5']/df['avg_pkatt_5'], 0
)
#Form indicator (Rolling win rate)
df['form_5']=np.nan
for team in df['team'].unique():
    team_mask=df['team']==team
    team_indices=df[team_mask].index
    #Percentage of wins in last 5 matches
    rolling_form=(
        df.loc[team_mask,'outcome'].shift(1).rolling(window=window_size, min_periods=1)
        .apply(lambda x: (x==1).sum()/len(x))
    )
    df.loc[team_indices,'form_5']=rolling_form.values
# Points from last 5 matchs (Win=3, Draw=1, Loss=0)
df['points_5']=np.nan
for team in df['team'].unique():
    team_mask=df['team']==team
    team_indices=df[team_mask].index
    #Convert outcome to points 
    points=df.loc[team_mask, 'outcome'].shift(1).map({1:3, 0:1, -1:0})
    rolling_points=points.rolling(window=window_size, min_periods=1).sum()
    df.loc[team_indices, 'points_5']=rolling_points.values
print("New features created: \n" 
"avg_goal_diff, \n" 
"avg_xgoal_diff, \n" 
"avg_shot_acc, \n" 
"avg_pk_acc, \n" 
"form_5, \n" 
"points_5")

New features created: 
avg_goal_diff, 
avg_xgoal_diff, 
avg_shot_acc, 
avg_pk_acc, 
form_5, 
points_5


# Encode categorical columns

In [27]:
# Encoding categorical variables
categorical_cols=['team', 'opponent', 'venue', 'formation', 'opp formation']

#Checking with columns exist
existing_categorical=[c for c in categorical_cols if c in df.columns]
print(f"Encoding {len(existing_categorical)} categorical columns")

#Storing label encoders
label_encoders={}
for col in existing_categorical:
    le=LabelEncoder()
    #Handling missing values by converting to string
    df[f"{col}_encoded"]=le.fit_transform(df[col].astype(str))
    label_encoders[col]=le
    n_unique=len(le.classes_)
    print(f" - {col}: {n_unique} unique values")

#Additional categorical encodings
if 'day_of_week' in df.columns:
    le_dow=LabelEncoder()
    df['day_of_week_encoded']=le_dow.fit_transform(df['day_of_week'].astype(str))
    label_encoders['day_of_week']=le_dow
    print(f" - day of week: {len(le_dow.classes_)} unique values")
#Encode home and away as binary 
if 'venue' in df.columns:
    df['is_home']=np.where(df['venue']=='Home',1,0)
    print("Created binary feature: is_home where Home=1, Away=0")


Encoding 5 categorical columns
 - team: 28 unique values
 - opponent: 28 unique values
 - venue: 2 unique values
 - formation: 22 unique values
 - opp formation: 21 unique values
 - day of week: 7 unique values
Created binary feature: is_home where Home=1, Away=0


# Select fetaures and drop data leakage columns 

In [28]:
# Selecting final features and removing data leakage 

leakage_columns = [
    'result',      
    'gf', 'ga',    
    'xg', 'xga',   
    'sh', 'sot',   
    'dist',        
    'fk',          
    'pk', 'pkatt', 
    'poss', 
]

keep_columns = [
    # Identifiers (keep for analysis, drop before training)
    'date', 'team', 'opponent', 'season', 'year', 'month', 'day_of_week',
    
    # Target variable
    'outcome',
    
    # Encoded categoricals
    'team_encoded', 'opponent_encoded', 'venue_encoded',
    'formation_encoded', 'opp formation_encoded',
    'is_home',
    
    # Rolling averages (last 5 matches)
    'avg_gf_5', 'avg_ga_5', 'avg_xg_5', 'avg_xga_5', 'avg_poss_5',
    'avg_sh_5', 'avg_sot_5', 'avg_dist_5', 'avg_fk_5', 'avg_pk_5', 'avg_pkatt_5',
    
    # Derived features (CORRECT NAMES)
    'avg_goal_diff', 'avg_xgoal_diff', 'avg_shot_acc', 
    'avg_pk_acc', 'form_5', 'points_5',
    
    # Context features
    'attendance', 'round', 'comp', 'time'
]

if 'day_of_week_encoded' in df.columns:
    keep_columns.append('day_of_week_encoded')

# Filter to only existing columns
keep_columns = [col for col in keep_columns if col in df.columns]

# Create the engineered dataset
engineered_df = df[keep_columns].copy()

# Drop rows with NaN in form/points (first match per team)
rows_before = len(engineered_df)
engineered_df = engineered_df.dropna(subset=['form_5', 'points_5']).reset_index(drop=True)
rows_after = len(engineered_df)

print(f"\nColumns removed due to data leakage:")
for col in leakage_columns:
    if col in df.columns:
        print(f" - {col}")

print(f"\nFeatures retained for modeling: {len(keep_columns)} columns")
print(f"Rows dropped (form/points NaN): {rows_before - rows_after}")
print(f"Final Shape: {engineered_df.shape}")

# Verify no NaNs in key features
print(f"\nNaN check:")
nan_cols = engineered_df.isna().sum()
nan_cols = nan_cols[nan_cols > 0]
if nan_cols.empty:
    print("No NaN values in retained features ✓")
else:
    print(nan_cols)

print(engineered_df.head())


Columns removed due to data leakage:
 - result
 - gf
 - ga
 - xg
 - xga
 - sh
 - sot
 - dist
 - fk
 - pk
 - pkatt
 - poss

Features retained for modeling: 36 columns
Rows dropped (form/points NaN): 28
Final Shape: (4644, 36)

NaN check:
No NaN values in retained features ✓
        date    team       opponent  season  year  month day_of_week  outcome  team_encoded  opponent_encoded  venue_encoded  formation_encoded  opp formation_encoded  is_home  avg_gf_5  avg_ga_5  avg_xg_5  avg_xga_5  avg_poss_5  avg_sh_5  avg_sot_5  avg_dist_5  avg_fk_5  avg_pk_5  avg_pkatt_5  avg_goal_diff  avg_xgoal_diff  avg_shot_acc  avg_pk_acc  form_5  points_5  attendance        round     comp   time  day_of_week_encoded
0 2019-08-31  Alaves         Getafe    2019  2019      8    Saturday        0             0                10              0                 20                     17        0  0.500000  0.000000  0.650000   0.650000       37.50      9.50   2.000000       18.45  0.500000       0.0          0.

# Display Summary

In [29]:
# Enhanced Engineered featured summary 
print(f"\nFinal dataset shape: {engineered_df.shape}")
print(f"\nNumber of features: {len(engineered_df.columns)}")
print(f"\nNumber of samples: {len(engineered_df)}")

print("\nTemporal features:")
temporal_features = [
    'date', 'season', 'year', 'month', 'day_of_week', 'day_of_week_encoded', 'season', 'time', 'round'
]   

for col in temporal_features:
    if col in engineered_df.columns:
        print(f" - {col}")

print("\nCategorical features (Encoded): ")
categorical_encoded=[col for col in engineered_df.columns if '_encoded' in col or col=='is_home']
for col in categorical_encoded:
    print(f" - {col}")

print("\nRolling Average Features (Basic):")
basic_rolling = ['avg_gf_5', 'avg_ga_5', 'avg_xg_5', 'avg_xga_5', 'avg_poss_5']
for col in basic_rolling:
    if col in engineered_df.columns:
        print(f"  - {col}")

print("\nRolling Average Features (Shots & Set Pieces):")
shot_rolling = ['avg_sh_5', 'avg_sot_5', 'avg_dist_5', 'avg_fk_5', 'avg_pk_5', 'avg_pkatt_5']
for col in shot_rolling:
    if col in engineered_df.columns:
        print(f"  - {col}")

print("\nDerived Performance Features:")
derived_features = ['avg_goal_diff', 'avg_xgoal_diff', 'avg_shot_acc', 
    'avg_pk_acc', 'form_5', 'points_5',
    ]
for col in derived_features:
    if col in engineered_df.columns:
        print(f"  - {col}")

print("\nOther Features:")
other_features = ['attendance']
for col in other_features:
    if col in engineered_df.columns:
        print(f"  - {col}")

print("\nSample of engineered features:")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(engineered_df.head(3))

print("\nDescriptive statistics of engineered features:")
print(engineered_df.describe().T)

print("\nTareget variable distribution (outcome):")
outcome_counts=engineered_df['outcome'].value_counts().sort_index()
print(outcome_counts)


Final dataset shape: (4644, 36)

Number of features: 36

Number of samples: 4644

Temporal features:
 - date
 - season
 - year
 - month
 - day_of_week
 - day_of_week_encoded
 - season
 - time
 - round

Categorical features (Encoded): 
 - team_encoded
 - opponent_encoded
 - venue_encoded
 - formation_encoded
 - opp formation_encoded
 - is_home
 - day_of_week_encoded

Rolling Average Features (Basic):
  - avg_gf_5
  - avg_ga_5
  - avg_xg_5
  - avg_xga_5
  - avg_poss_5

Rolling Average Features (Shots & Set Pieces):
  - avg_sh_5
  - avg_sot_5
  - avg_dist_5
  - avg_fk_5
  - avg_pk_5
  - avg_pkatt_5

Derived Performance Features:
  - avg_goal_diff
  - avg_xgoal_diff
  - avg_shot_acc
  - avg_pk_acc
  - form_5
  - points_5

Other Features:
  - attendance

Sample of engineered features:
        date    team       opponent  season  year  month day_of_week  outcome  team_encoded  opponent_encoded  venue_encoded  formation_encoded  opp formation_encoded  is_home  avg_gf_5  avg_ga_5  avg_xg_5  a

# Saving The Engineered Dataset 

In [31]:
output_path=os.path.join("..","Processed", "engineered_data.csv")
engineered_df.to_csv(output_path, index=False)
print(f"\nEngineered features saved to: {output_path}")
print(f"\nFinal dataset shape: {engineered_df.shape}")

#Saving encoding information for future use
encoding_info=[]
for col, encoder in label_encoders.items():
    for i, class_label in enumerate(encoder.classes_):
        encoding_info.append({
            'column': col,
            'original_value': class_label,
            'encoded_value': i
        })
encoding_df=pd.DataFrame(encoding_info)
encoding_output_path=os.path.join("..","Processed","label_encodings_data.csv")
encoding_df.to_csv(encoding_output_path, index=False)
print(f"Label encodings saved to: {encoding_output_path}")



Engineered features saved to: ..\Processed\engineered_data.csv

Final dataset shape: (4644, 36)
Label encodings saved to: ..\Processed\label_encodings_data.csv
